# Financial Analyst Agent — End-to-End Demo

This notebook walks through the complete MVP for an agentic AI interview demo.

**Stack**: LangGraph · Azure AI Search (Basic) · Azure OpenAI (gpt-4o + text-embedding-3-small)

**Documents**: JPMorgan 10-K · Fed FSR · Basel III · ECB Economic Bulletin

---

## Table of Contents
1. [Setup & Configuration](#1-setup--configuration)
2. [Chunking Strategy Comparison](#2-chunking-strategy-comparison)
3. [Hybrid Search Demo](#3-hybrid-search-demo)
4. [Agent Graph Visualization](#4-agent-graph-visualization)
5. [Full Agent Run](#5-full-agent-run)
6. [Final Report Output](#6-final-report-output)

## 1. Setup & Configuration

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

from src.config import settings
print('Azure OpenAI endpoint:', settings.azure_openai_endpoint)
print('Azure Search endpoint:', settings.azure_search_endpoint)
print('Index name:', settings.azure_search_index_name)

## 2. Chunking Strategy Comparison

This is the key interview talking point: **three strategies, one document, dramatically different results**.

We'll chunk the JPMorgan 10-K with all three strategies and compare:
- `fixed_size`: baseline, naive token windows
- `semantic`: embedding-based topic boundaries
- `hierarchical`: finance-document-aware (recommended)

In [ ]:
from src.ingestion.pdf_parser import PDFParser
from src.ingestion.chunking import FixedSizeChunker, HierarchicalChunker

pdf_path = PROJECT_ROOT / 'data' / 'pdfs' / 'jpmorgan_10k_2023.pdf'

print('Parsing JPMorgan 10-K...')
parser = PDFParser()
parsed_doc = parser.parse(pdf_path, document_id='jpmorgan_10k_2023')
print(f'Pages: {parsed_doc.total_pages}, Blocks: {len(parsed_doc.blocks)}')

In [ ]:
# Run both strategies (skip semantic to avoid embedding costs in demo)
fixed_chunker = FixedSizeChunker(max_tokens=512)
hier_chunker = HierarchicalChunker(max_tokens=512)

print('Fixed-size chunking...')
fixed_chunks = fixed_chunker.chunk(parsed_doc)
print(f'  → {len(fixed_chunks)} chunks')

print('Hierarchical chunking...')
hier_chunks = hier_chunker.chunk(parsed_doc)
print(f'  → {len(hier_chunks)} chunks')

In [ ]:
# Comparison metrics
import pandas as pd

def chunk_metrics(chunks, name):
    prose = [c for c in chunks if c.chunk_type == 'prose']
    tables = [c for c in chunks if c.chunk_type == 'table']
    captions = [c for c in chunks if c.chunk_type == 'caption']
    footnotes = [c for c in chunks if c.chunk_type == 'footnote']
    with_section = [c for c in prose if c.section_path]
    avg_tokens = sum(c.token_count for c in chunks) / len(chunks) if chunks else 0
    
    return {
        'Strategy': name,
        'Total Chunks': len(chunks),
        'Avg Tokens': round(avg_tokens),
        'Table Chunks': len(tables),
        'Caption Chunks': len(captions),
        'Footnote Chunks': len(footnotes),
        'Section Path %': f"{round(100*len(with_section)/len(prose)) if prose else 0}%",
    }

df = pd.DataFrame([
    chunk_metrics(fixed_chunks, 'fixed_size'),
    chunk_metrics(hier_chunks, 'hierarchical'),
])
df.set_index('Strategy')

In [ ]:
# Show a sample hierarchical table chunk
table_chunks = [c for c in hier_chunks if c.chunk_type == 'table']
if table_chunks:
    sample = table_chunks[0]
    print('=== Sample Table Chunk (Hierarchical) ===')
    print(f'Section: {sample.section_path_str or "(root)"}')
    print(f'Page: {sample.page_start}')
    print(f'Context (preceding prose): {sample.table_context[:200]}...')
    print(f'Content:')
    print(sample.text[:600])
    print()

# Show what fixed_size does to the same content
print('=== Fixed-Size: No dedicated table chunks ===')
f_tables = [c for c in fixed_chunks if c.chunk_type == 'table']
print(f'Table chunks in fixed_size: {len(f_tables)} (expected 0 — tables are split into prose stream)')

## 3. Hybrid Search Demo

Shows the three-stage retrieval pipeline:
1. **BM25** — exact match on financial jargon  
2. **Vector ANN (HNSW)** — semantic recall  
3. **Semantic reranking** — Azure cross-encoder model

We compare vector-only vs hybrid on a domain-specific query.

In [ ]:
from src.retrieval.search_client import FinancialSearchClient

client = FinancialSearchClient()
query = 'What is JPMorgan Tier 1 capital ratio CET1 requirement'

print(f'Query: {query}\n')
results = client.hybrid_search(query, document_type_filter='10k', top_k=5)

for i, r in enumerate(results, 1):
    print(f'Result {i}:')
    print(f'  Doc: {r["document_id"]} | Page: {r["page_start"]} | Type: {r["chunk_type"]}')
    print(f'  Section: {r.get("section_path_str", "")}')  
    print(f'  Reranker score: {r.get("reranker_score", "N/A")}')
    print(f'  Text: {r["text"][:200]}...')
    print()

In [ ]:
# Table-targeted search: retrieve a specific financial table
print('=== Table-targeted search: Basel III capital requirements table ===')
table_results = client.table_search(
    'minimum capital requirements CET1 tier 1 leverage ratio',
    document_type_filter='basel3',
    top_k=3,
)

for i, r in enumerate(table_results, 1):
    print(f'Table result {i}: {r["document_id"]} p.{r["page_start"]}')
    print(r['text'][:400])
    print()

## 4. Agent Graph Visualization

LangGraph provides built-in Mermaid diagram generation.  
The graph shows all nodes, edges, and conditional routing logic.

In [ ]:
from src.agent.graph import build_graph

graph = build_graph()

# Print as Mermaid (paste at mermaid.live to visualize)
try:
    mermaid_diagram = graph.get_graph().draw_mermaid()
    print(mermaid_diagram)
except Exception as e:
    print(f'Mermaid generation error: {e}')
    # Fallback: print static diagram
    print('''
flowchart TD
    START([User Query]) --> decompose_query
    decompose_query -->|valid JSON| retrieve_context
    decompose_query -->|malformed, retry<3| decompose_query
    decompose_query -->|retry>=3| END
    retrieve_context -->|more sub-questions pending| retrieve_context
    retrieve_context -->|all sub-questions done| evaluate_sufficiency
    evaluate_sufficiency -->|score >= 0.6| synthesize_report
    evaluate_sufficiency -->|score < 0.6, retry<3| refine_retrieval
    evaluate_sufficiency -->|retry>=3| synthesize_report
    refine_retrieval --> retrieve_context
    synthesize_report --> format_output
    format_output --> END([Final Report + Citations])
    ''')

## 5. Full Agent Run

Run the complete multi-step agent on a complex financial query.  
Watch each node execute in sequence with verbose output.

In [ ]:
from src.agent.state import initial_state

QUERY = "What are the key credit risk factors for large US banks and how does Basel III address them?"

print(f'Query: {QUERY}\n')
print('Running agent...\n')

state = initial_state(QUERY)
final_state = None

# Stream execution step by step
for step in graph.stream(state):
    node_name = list(step.keys())[0]
    node_output = step[node_name]
    print(f'── {node_name}')
    
    if node_name == 'decompose_query':
        sqs = node_output.get('sub_questions', [])
        print(f'   Sub-questions ({len(sqs)}):')
        for sq in sqs:
            scope = f" [{sq['document_scope']}]" if sq.get('document_scope') else ''
            print(f'   {sq["id"]}: {sq["question"][:70]}{scope}')
    
    elif node_name == 'retrieve_context':
        idx = node_output.get('current_sq_index', 0)
        print(f'   Processed sub-question, index now: {idx}')
    
    elif node_name == 'evaluate_sufficiency':
        score = node_output.get('overall_sufficiency', 0)
        reasons = node_output.get('insufficiency_reasons', [])
        print(f'   Sufficiency score: {score:.2f}')
        if reasons:
            print(f'   Issues: {len(reasons)}')
    
    elif node_name == 'refine_retrieval':
        print('   Reformulating sub-questions...')
    
    elif node_name == 'synthesize_report':
        draft = node_output.get('draft_report', '')
        print(f'   Draft report: {len(draft)} characters')
    
    elif node_name == 'format_output':
        final_report = node_output.get('final_report', '')
        print(f'   Final report: {len(final_report)} characters')
    
    final_state = {**state, **node_output}
    print()

## 6. Final Report Output

The final report includes:
- Executive Summary (3-5 bullets)
- Thematic analysis sections
- Regulatory-Risk Cross-Reference table
- Key Quantitative Metrics
- References with page numbers and section paths

In [ ]:
from IPython.display import Markdown, display

if final_state and final_state.get('final_report'):
    display(Markdown(final_state['final_report']))
else:
    print('No final report generated. Run Cell 5 first.')

In [ ]:
# Agent execution summary
if final_state:
    sqs = final_state.get('sub_questions', [])
    print('=== Agent Execution Summary ===')
    print(f'Sub-questions: {len(sqs)}')
    print(f'Total chunks retrieved: {final_state.get("total_chunks_retrieved", 0)}')
    print(f'Overall sufficiency: {final_state.get("overall_sufficiency", 0):.2f}')
    print(f'Iterations: {final_state.get("iteration_count", 0)}')
    print()
    print('Per sub-question scores:')
    for sq in sqs:
        scope = f" [{sq.get('document_scope', 'all')}]" if sq.get('document_scope') else ''
        print(f'  {sq["id"]}{scope}: score={sq.get("retrieval_score", 0):.2f}, retries={sq.get("retry_count", 0)}')